In [ ]:
import numpy as np

from scripts.data_loader import load_dataframe
from scripts.data_writer import save_experiment_result
from scripts.utils import NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

In [ ]:
from scripts.data_filter import filter_dataframe

df = filter_dataframe(
    df=df,
    operators=[10],
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq']
)

In [ ]:
from scripts.weighted_coverage import run_weighted_coverage
from scripts.utils import extract_unique_npcis, RF_PARAM_5G
import pandas as pd

# Parameters
n_runs = 1
k_wknn = 2
rf_params = [RF_PARAM_5G.RSRQ]
clustering_params = [RF_PARAM_5G.RSRQ]
operator_choice = [10]
cluster_range = range(0, 1)

print(f"""
Running weighted coverage

🧪 Experiment setup 🧪
🔢 k-value for wKNN = {k_wknn}
👨‍👩‍👦‍👦 cluster range = {cluster_range[0]} - {cluster_range[-1]}
🛜 RF PARAM {str(rf_params)}
📶 Operator choice {operator_choice}
🔁 Number of runs {n_runs}|
_________________________________
""")

unique_npcis = extract_unique_npcis(df['measurements_matrix'])
errors_dict = {c: [] for c in cluster_range}  # Store the errors
complexity_dict = {c: [] for c in cluster_range}
runtime_dict = {c: [] for c in cluster_range}

for c in cluster_range:
    for i in range(n_runs):
        print(f"\r🔄 Running for {c} clusters ({i + 1}/{n_runs} runs)", end="")
        random = random_seeds[i]

        _, errors, complexity, runtime = run_weighted_coverage(
            df=df,
            rf_params=rf_params,
            cluster_rf_params=clustering_params,
            k_max=k_wknn,
            unique_npcis=unique_npcis,
            random_seed=random,
            n_clusters=c,
        )
        errors_dict[c].append(errors.mean())
        complexity_dict[c].append(complexity)
        runtime_dict[c].append(runtime)

    print(f"\r✅ {c} completed                                           ")

errors_df = pd.DataFrame(errors_dict)
complexity_df = pd.DataFrame(complexity_dict)
runtime_df = pd.DataFrame(runtime_dict)

In [ ]:
config = {
    'wknn_k': k_wknn,
    'rf_param': rf_params[0].value,
    'cluster_rf_param': clustering_params[0].value,
    'operator_choice': operator_choice,
    'cluster_range': list(cluster_range),
    'n_runs': n_runs,
}

data = {
    'errors': errors_df,
    'complexity': complexity_df,
    'runtime': runtime_df,
}

save_experiment_result('experiment3-clustering', config, data)

In [ ]:
from scripts.plotting import make_boxplot
from scripts.utils import params_to_str
from scripts.utils import operators_to_str

operator_choice = np.array(operator_choice)

# Plotting with capped runtime data
param_str = f'RF Params: {params_to_str(rf_params)}'
operator_str = f'Operators: {operators_to_str(operator_choice)}'
title_string = f"wKNN with k-means clustering \n {param_str} * {operator_str} * {n_runs} runs,"

make_boxplot(
    df=errors_df,
    title=f'Errors: {title_string}',
    x_label='Cluster size for k-means',
    y_label='Error (m)',
    color='mediumseagreen',
)

make_boxplot(
    df=runtime_df,
    title=f'Runtime: {title_string}',
    x_label='Cluster size for k-means',
    y_label='Runtime (s)',
    color='lightskyblue',
)

make_boxplot(
    df=complexity_df,
    title=f'Complexity: {title_string}',
    x_label='Cluster size for k-means',
    y_label='Complexity metric',
    color='plum',
)